In [7]:
import json
import random
import torch
from datasets import Dataset
from huggingface_hub import notebook_login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model


In [ ]:
# --- GLOBALS ---
# File paths
JSON_FILE = "better_training_data.json"
OUTPUT_DIR = "gemma"

# Model ID
MODEL_ID = "google/gemma-3-1b-pt"

# Data hyperparameters
SEED = 42
SUBSET_SIZE = 50000
MAX_SEQ_LENGTH = 512

# Training hyperparameters
NUM_EPOCHS = 4
LEARNING_RATE = 2e-4
BATCH_SIZE = 8
GRAD_ACCUMULATION = 2
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
NUM_WORKERS = 8

In [9]:
# ---LOAD DATA ---
random.seed(SEED)

with open(JSON_FILE, "r") as f:
    data = json.load(f)

# Take a random subset
subset_data = random.sample(data, min(SUBSET_SIZE, len(data)))
records = [{"input_text": d["input_text"], "output_text": d["output_text"]} for d in subset_data]
dataset = Dataset.from_list(records)

print(f"Loaded {len(dataset)} examples.")

# --- AUTHENTICATE AND LOAD TOKENIZER ---
notebook_login() # Added to authenticate with Hugging Face
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# --- TOKENIZATION FUNCTION (SIMPLIFIED) ---
def tokenize_fn(examples):
    model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for inp, out in zip(examples["input_text"], examples["output_text"]):
        # FORMAT: "Input \n Output"
        # We want the model to read the input and predict the output immediately.
        # No complex instructions.
        prompt = f"{inp}\n"
        completion = f"{out}"
        full_text = prompt + completion + tokenizer.eos_token

        # Tokenize full text
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding="max_length",
        )

        input_ids = tokenized["input_ids"]
        labels = input_ids.copy()

        # Calculate length of the prompt part to mask it
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        prompt_len = len(prompt_ids)

        # Mask the prompt (so we don't train on identifying the input, only generating output)
        for i in range(len(labels)):
            if i < prompt_len:
                labels[i] = -100  # Ignore prompt in loss
            elif labels[i] == tokenizer.pad_token_id:
                labels[i] = -100  # Ignore padding in loss

        model_inputs["input_ids"].append(input_ids)
        model_inputs["attention_mask"].append(tokenized["attention_mask"])
        model_inputs["labels"].append(labels)

    return model_inputs

Loaded 50000 examples.


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [10]:
# Apply tokenization
tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

# --- LOAD MODEL & APPLY LoRA ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

trainable params: 745,472 || all params: 1,000,631,424 || trainable%: 0.0745


In [11]:
# --- SETUP TRAINER ---
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=True,
    optim="adamw_torch",
    report_to="none",
    dataloader_num_workers=NUM_WORKERS,
    group_by_length=True,
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [12]:
# --- TRAIN AND SAVE MODEL ---
print("Starting fine-tuning...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Fine-tuning complete. Model saved to {OUTPUT_DIR}")

Starting fine-tuning...


Step,Training Loss
10,2.404300
20,1.952400
30,1.517000
40,1.282000
50,1.195700
60,1.116100
70,1.055300
80,1.059700
90,1.024700
100,1.011300


Fine-tuning complete. Model saved to /content/drive/MyDrive/BetaGeometry/Gemma


In [19]:
# --- MODIFIED INFERENCE (Fixing the Warning) ---
model.eval()
raw_prompt = "circle X1 X2 X3 X5; circle X1 X2 X3 X6; midp X7 X3 X4; perp X5 X4 X2 X7; col X2 X5 X7; col X3 Q X4; midp X8 X2 X6; ? cong X8 X3 X8 X4"
formatted_prompt = f"{raw_prompt}\n"  # Just add the newline we trained on

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

# Inference code
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 1. Remove the prompt (Gemma 3 might echo it)
gen_text = raw_output[len(raw_prompt):].strip()

# 2. Extract only the first line
first_line = gen_text.split('\n')[0]

print(f"Input Prompt: {raw_prompt}")
# 3. Validation (Optional)
if "(" in first_line and first_line.endswith(")"):
    print(f"Valid Construction: {first_line}")
else:
    # If it generated multiple calls on one line like "midp(A,B) perp(C,D)", take the first
    first_call = first_line.split(")")[0] + ")"
    print(f"Sanitized Construction: {first_call}")

Input Prompt: circle X1 X2 X3 X5; circle X1 X2 X3 X6; midp X7 X3 X4; perp X5 X4 X2 X7; col X2 X5 X7; col X3 Q X4; midp X8 X2 X6; ? cong X8 X3 X8 X4
Sanitized Construction: excenter(X5, X8, X2)


In [ ]:
!zip -r /content/gemma.zip /content/gemma

Fine-tuning complete. Model saved to gemma
  adding: content/gemma/ (stored 0%)
  adding: content/gemma/adapter_model.safetensors (deflated 8%)
  adding: content/gemma/tokenizer.model (deflated 52%)
  adding: content/gemma/special_tokens_map.json (deflated 73%)
  adding: content/gemma/added_tokens.json (stored 0%)
  adding: content/gemma/adapter_config.json (deflated 57%)
  adding: content/gemma/tokenizer.json (deflated 83%)
  adding: content/gemma/training_args.bin (deflated 53%)
  adding: content/gemma/tokenizer_config.json (deflated 97%)
  adding: content/gemma/README.md (deflated 65%)
